## File summary

In [1]:
import pandas as pd
import pathlib

pd.set_option('display.max_colwidth', None)

In [2]:
path = pathlib.Path('../data/raw/csv').glob('*.csv')
data_info = []
for f in path:
    df = pd.read_csv(f, low_memory=False)
    data_info.append({
        "name" :f.name, "num_of_records": len(df),"num_of_cols":df.shape[1], "size [MB]" : pathlib.Path.stat(f).st_size/1000000,
        "full_duplicates": df.duplicated().sum()
    })

In [3]:
data_info = pd.DataFrame(data= data_info)
data_info.sort_values(by=['size [MB]'], ascending=False)

,name,num_of_records,num_of_cols,size [MB],full_duplicates
8,observations.csv,2328638,9,402.325980,518
13,procedures.csv,454904,10,92.237842,0
4,encounters.csv,193633,15,64.096185,0
5,imaging_studies.csv,117837,13,41.301920,0
7,medications.csv,111345,13,29.407940,0
2,conditions.csv,105497,7,15.438895,0
6,immunizations.csv,61740,6,8.502593,0
12,payer_transitions.csv,41902,8,7.140926,0
15,supplies.csv,21235,6,3.157260,187
1,careplans.csv,10098,9,2.001688,0


- 16 tables, \~667 MB in total; `observations` alone is \~60% of the size.
- Full duplicates appear only in `observations` (518) and `supplies` (187).

## Patients

In [4]:
patients = pd.read_csv('../data/raw/csv/patients.csv')
print(patients.shape)
patients.head(1)

(1154, 28)


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,ba419d35-0dfe-8af7-347c-eebf02485a56,1925-09-30,2019-04-22,999-49-6110,S99986871,X5350127X,Mrs.,Corie618,Sofia418,Mante251,...,Franklin,Massachusetts,Norfolk County,25021.0,2038,42.055928,-71.367756,141917.67,2367826.28,18756


In [5]:
patients.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   1154 non-null   str    
 1   BIRTHDATE            1154 non-null   str    
 2   DEATHDATE            154 non-null    str    
 3   SSN                  1154 non-null   str    
 4   DRIVERS              954 non-null    str    
 5   PASSPORT             885 non-null    str    
 6   PREFIX               915 non-null    str    
 7   FIRST                1154 non-null   str    
 8   MIDDLE               917 non-null    str    
 9   LAST                 1154 non-null   str    
 10  SUFFIX               11 non-null     str    
 11  MAIDEN               307 non-null    str    
 12  MARITAL              745 non-null    str    
 13  RACE                 1154 non-null   str    
 14  ETHNICITY            1154 non-null   str    
 15  GENDER               1154 non-null   str    
 16 

In [6]:
patients['Id'].is_unique

True

In [7]:
patients[['FIPS','COUNTY']].drop_duplicates()

,FIPS,COUNTY
0,25021.0,Norfolk County
6,NaN,Plymouth County
7,25017.0,Middlesex County
9,25009.0,Essex County
11,25025.0,Suffolk County
12,25027.0,Worcester County
21,25005.0,Bristol County
29,NaN,Worcester County
35,NaN,Barnstable County
36,NaN,Bristol County


In [8]:
patients[patients['FIPS'] == 25017]['COUNTY'].unique()

<StringArray>
['Middlesex County', 'Suffolk County', 'Norfolk County']
Length: 3, dtype: str

In [9]:
patients[patients['FIPS'].isna()]['Id'].nunique()

297

In [10]:
patients['GENDER'].value_counts()

GENDER
F    589
M    565
Name: count, dtype: int64

In [11]:
patients['ZIP'].astype(str).str.len().value_counts()

ZIP
4    857
1    297
Name: count, dtype: int64

In [12]:
((patients['ZIP'] == 0) == patients['FIPS'].isna()).all()

np.True_

**One row:** one patient.

**Missing values:**
- DEATHDATE (patient alive), DRIVERS, PASSPORT, PREFIX, MIDDLE,
  SUFFIX, MAIDEN, MARITAL (optional or age-dependent fields).
- FIPS is missing in 297 rows although COUNTY is always filled.
  FIPS and COUNTY also disagree in some rows (e.g. FIPS `25017` appears with
  "Middlesex County", "Suffolk County" and "Norfolk County"; FIPS `44007`
  appears with "Bristol County").

**Note:**
- ZIP loads as a number, so leading zeros are lost (857 ZIPs have 4 digits
  instead of 5). ZIP needs to be stored as text.
- 297 patients have ZIP `0`; these are exactly the rows with missing FIPS.

## Encounters

In [13]:
encounters = pd.read_csv('../data/raw/csv/encounters.csv')
print(encounters.shape)
encounters.head(1)

(193633, 15)


,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
0,ba419d35-0dfe-8af7-f9e0-52523777fdca,1925-09-30T16:40:18Z,1925-09-30T16:55:18Z,ba419d35-0dfe-8af7-347c-eebf02485a56,4705a8fd-19cd-32c1-8b3e-34bcfe84d0bd,95e71205-f63b-3e56-adf3-99bc2c9f4550,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.8,427.08,277.08,NaN,NaN


In [14]:
encounters.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 193633 entries, 0 to 193632
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Id                   193633 non-null  str    
 1   START                193633 non-null  str    
 2   STOP                 193633 non-null  str    
 3   PATIENT              193633 non-null  str    
 4   ORGANIZATION         193633 non-null  str    
 5   PROVIDER             193633 non-null  str    
 6   PAYER                193633 non-null  str    
 7   ENCOUNTERCLASS       193633 non-null  str    
 8   CODE                 193633 non-null  int64  
 9   DESCRIPTION          193633 non-null  str    
 10  BASE_ENCOUNTER_COST  193633 non-null  float64
 11  TOTAL_CLAIM_COST     193633 non-null  float64
 12  PAYER_COVERAGE       193633 non-null  float64
 13  REASONCODE           117040 non-null  float64
 14  REASONDESCRIPTION    117040 non-null  str    
dtypes: float64(4), int64(1), str

In [15]:
encounters['Id'].is_unique

True

In [16]:
print("CODES:",encounters['CODE'].nunique())
print("DESCRIPTIONS:",encounters['DESCRIPTION'].nunique())
print("Code-description pairs:", encounters[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 57
DESCRIPTIONS: 57
Code-description pairs: 57


In [17]:
encounters['ENCOUNTERCLASS'].value_counts()

ENCOUNTERCLASS
ambulatory    105287
wellness       46620
outpatient     21856
emergency       8981
urgentcare      6827
inpatient       2138
home             758
snf              615
virtual          335
hospice          216
Name: count, dtype: int64

**One row:** one encounter.

**Missing values:** REASONCODE and REASONDESCRIPTION are each missing in \~40% of rows.

**Codes:** 57 codes, each with exactly one DESCRIPTION.

## Conditions

In [18]:
conditions = pd.read_csv('../data/raw/csv/conditions.csv')
print(conditions.shape)
conditions.head(1)

(105497, 7)


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
0,1925-09-30,1925-09-30,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-f9e0-52523777fdca,SNOMED-CT,314529007,Medication review due (situation)


In [19]:
conditions.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 105497 entries, 0 to 105496
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   START        105497 non-null  str  
 1   STOP         94954 non-null   str  
 2   PATIENT      105497 non-null  str  
 3   ENCOUNTER    105497 non-null  str  
 4   SYSTEM       105497 non-null  str  
 5   CODE         105497 non-null  str  
 6   DESCRIPTION  105497 non-null  str  
dtypes: str(7)
memory usage: 5.6 MB


In [20]:
conditions.duplicated(subset=['CODE','ENCOUNTER','PATIENT']).sum()

np.int64(0)

In [21]:
print("Rows repeating a patient + code pair:", conditions.duplicated(subset=['PATIENT', 'CODE']).sum())
print("Fully duplicated rows:", conditions.duplicated().sum())

Rows repeating a patient + code pair: 71768
Fully duplicated rows: 0


In [22]:
dup = conditions[conditions.duplicated(subset=['PATIENT', 'CODE'])]
dup['DESCRIPTION'].value_counts().head(10)

DESCRIPTION
Medication review due (situation)     28026
Stress (finding)                       6372
Gingivitis (disorder)                  6064
Full-time employment (finding)         5631
Viral sinusitis (disorder)             4142
Part-time employment (finding)         3583
Acute viral pharyngitis (disorder)     2099
Normal pregnancy (finding)             2079
Social isolation (finding)             1902
Limited social contact (finding)       1864
Name: count, dtype: int64

In [23]:
print('Codes:', conditions['CODE'].nunique())
print('Descriptions:', conditions['DESCRIPTION'].nunique())
print('Code-description pairs:', conditions[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

Codes: 493
Descriptions: 483
Code-description pairs: 494


In [24]:
conditions.groupby('CODE')['DESCRIPTION'].nunique().sort_values(ascending=False).head(3)

CODE
81629009            2
1085951000119102    1
1085961000119100    1
Name: DESCRIPTION, dtype: int64

In [25]:
conditions[conditions['CODE'] == '81629009']['DESCRIPTION'].unique()

<StringArray>
['Traumatic dislocation of temporomandibular joint (disorder)', 'Traumatic dislocation of temporomandibular joint']
Length: 2, dtype: str

In [26]:
conditions.groupby('DESCRIPTION')['CODE'].nunique().sort_values(ascending=False).head(15)

DESCRIPTION
Dental caries on smooth surface penetrating into pulp                             2
Dental caries on pit and fissure surface penetrating into pulp                    2
Cracked tooth                                                                     2
Gingival recession                                                                2
Dental caries on smooth surface penetrating into dentin                           2
Gingival enlargement                                                              2
Arrested dental caries                                                            2
Fractured dental restorative material with loss of material                       2
Dental caries on pit and fissure surface limited to enamel                        2
Dental caries on pit and fissure surface penetrating into dentin                  2
Retained dental root                                                              2
Acute pulmonary embolism (disorder)                             

In [27]:
conditions[conditions['DESCRIPTION'] == 'Cracked tooth'][['SYSTEM','CODE']].drop_duplicates()

,SYSTEM,CODE
4237,ICD10,K03.81
9566,SNOMED-CT,109747007


In [28]:
conditions['SYSTEM'].value_counts()

SYSTEM
SNOMED-CT    104391
ICD10          1106
Name: count, dtype: int64

In [29]:
conditions[conditions['SYSTEM'] == 'ICD10']['CODE'].nunique()

195

In [30]:
conditions.groupby('CODE')['SYSTEM'].nunique().sort_values(ascending=False).head()

CODE
10509002            1
105531004           1
105995000           1
1085741000119102    1
1085951000119102    1
Name: SYSTEM, dtype: int64

**One row:** one condition recorded for one patient at one encounter
(`PATIENT, ENCOUNTER, CODE` is unique). The same condition can repeat across
encounters: \~68% of rows repeat a PATIENT + CODE pair, so row counts overstate
the number of affected patients.

**Missing values:** STOP is missing in 10,543 rows (condition
still ongoing).

**Codes:**
- \~1% of rows (195 codes) use ICD10 instead of SNOMED-CT; 11 dental conditions
  exist in both.
- CODE `81629009` has two descriptions, with and without the `(disorder)` tag.

**Note:** DESCRIPTION mixes diseases `(disorder)`, findings and administrative
situations; the most frequent entry is "Medication review due (situation)".

## Medications

In [31]:
medications = pd.read_csv('../data/raw/csv/medications.csv')
print(medications.shape)
medications.head(1)

(111345, 13)


,START,STOP,PATIENT,PAYER,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,PAYER_COVERAGE,DISPENSES,TOTALCOST,REASONCODE,REASONDESCRIPTION
0,1929-03-16T15:40:18Z,1929-03-28T04:40:18Z,ba419d35-0dfe-8af7-347c-eebf02485a56,df166300-5a78-3502-a46a-832842197811,ba419d35-0dfe-8af7-4143-c917ecb93b9a,834061,Penicillin V Potassium 250 MG Oral Tablet,158.22,108.22,1,158.22,43878008.0,Streptococcal sore throat (disorder)


In [32]:
medications.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 111345 entries, 0 to 111344
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   START              111345 non-null  str    
 1   STOP               108315 non-null  str    
 2   PATIENT            111345 non-null  str    
 3   PAYER              111345 non-null  str    
 4   ENCOUNTER          111345 non-null  str    
 5   CODE               111345 non-null  int64  
 6   DESCRIPTION        111345 non-null  str    
 7   BASE_COST          111345 non-null  float64
 8   PAYER_COVERAGE     111345 non-null  float64
 9   DISPENSES          111345 non-null  int64  
 10  TOTALCOST          111345 non-null  float64
 11  REASONCODE         88922 non-null   float64
 12  REASONDESCRIPTION  88922 non-null   str    
dtypes: float64(4), int64(2), str(7)
memory usage: 11.0 MB


In [33]:
medications.duplicated(subset=['START','PATIENT','ENCOUNTER','CODE']).sum()

np.int64(328)

In [34]:
medications.duplicated(subset=['START','STOP','PATIENT','ENCOUNTER','CODE']).sum()

np.int64(0)

In [35]:
dup = medications[medications.duplicated(subset=['START','PATIENT','ENCOUNTER','CODE'], keep=False)]
dup.groupby(['START','PATIENT','ENCOUNTER','CODE']).nunique().max()

STOP                 2
PAYER                1
DESCRIPTION          1
BASE_COST            2
PAYER_COVERAGE       2
DISPENSES            2
TOTALCOST            2
REASONCODE           1
REASONDESCRIPTION    1
dtype: int64

In [36]:
print("CODES:", medications['CODE'].nunique())
print("DESCRIPTIONS:", medications['DESCRIPTION'].nunique())
print("Codes-description", medications[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 267
DESCRIPTIONS: 278
Codes-description 278


In [37]:

n_desc = medications.groupby('CODE')['DESCRIPTION'].nunique()
multi_codes = n_desc[n_desc > 1].index
medications[medications['CODE'].isin(multi_codes)][['CODE','DESCRIPTION']].drop_duplicates().sort_values(['CODE','DESCRIPTION'],ascending=False)

,CODE,DESCRIPTION
20808,1804799,alteplase 100 MG Injection
11570,1804799,Alteplase 100 MG Injection
23272,1659149,piperacillin 4000 MG / tazobactam 500 MG Injection
90500,1659149,Piperacillin 4000 MG / tazobactam 500 MG Injection
3256,1049221,Acetaminophen 325 MG / oxyCODONE Hydrochloride 5 MG Oral Tablet
794,1049221,Acetaminophen 325 MG / Oxycodone Hydrochloride 5 MG Oral Tablet
4832,1000126,1 ML medroxyprogesterone acetate 150 MG/ML Injection
7123,1000126,1 ML medroxyPROGESTERone acetate 150 MG/ML Injection
4300,856987,Acetaminophen 300 MG / Hydrocodone Bitartrate 5 MG Oral Tablet
3091,856987,Acetaminophen 300 MG / HYDROcodone Bitartrate 5 MG Oral Tablet


**One row:** one medication order for one patient
(`PATIENT, ENCOUNTER, CODE, START, STOP` is unique). Without STOP the key is not unique: 328 rows share PATIENT, ENCOUNTER, CODE and
START. Within these rows STOP, BASE_COST, PAYER_COVERAGE, TOTALCOST and DISPENSES
can differ; all other columns are the same.

**Missing values:** STOP is missing in 3,030 rows (drug still
taken); REASONCODE and REASONDESCRIPTION are each missing in \~20% of rows.

**Codes:** 267 codes, 278 descriptions. 11 codes have two descriptions that
differ only in letter case (e.g. "aspirin" / "Aspirin", "oxyCODONE" / "Oxycodone").

## Observations

In [38]:
observations = pd.read_csv('../data/raw/csv/observations.csv')

print(observations.shape)
observations.head(1)

(2328638, 9)


,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
0,1925-09-30T16:40:18Z,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-f9e0-52523777fdca,vital-signs,8302-2,Body Height,52.0,cm,numeric


In [39]:
observations.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2328638 entries, 0 to 2328637
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   DATE         2328638 non-null  str  
 1   PATIENT      2328638 non-null  str  
 2   ENCOUNTER    2184557 non-null  str  
 3   CATEGORY     2184557 non-null  str  
 4   CODE         2328638 non-null  str  
 5   DESCRIPTION  2328638 non-null  str  
 6   VALUE        2328638 non-null  str  
 7   UNITS        1708452 non-null  str  
 8   TYPE         2328638 non-null  str  
dtypes: str(9)
memory usage: 159.9 MB


In [40]:
observations[['CODE','ENCOUNTER','PATIENT','DATE']].duplicated().sum()

np.int64(4756)

In [41]:
dup = observations[observations.duplicated(subset=['CODE','ENCOUNTER','PATIENT','DATE'], keep=False)]
dup.groupby(['CODE','ENCOUNTER','PATIENT','DATE'],dropna=False).nunique().max()

CATEGORY       1
DESCRIPTION    1
VALUE          6
UNITS          2
TYPE           1
dtype: int64

In [42]:
dup['DESCRIPTION'].value_counts().head(10)

DESCRIPTION
Glomerular filtration rate [Volume Rate/Area] in Serum or Plasma by Creatinine-based formula (MDRD)/1.73 sq M    6921
Oxygen saturation in Arterial blood                                                                               262
Appearance of Urine                                                                                               200
Procedure Narrative                                                                                               172
Glucose [Mass/volume] in Blood                                                                                    144
Urea nitrogen [Mass/volume] in Blood                                                                              144
Creatinine [Mass/volume] in Blood                                                                                 144
Calcium [Mass/volume] in Blood                                                                                    144
Sodium [Moles/volume] in Blood              

In [43]:
observations[observations['DESCRIPTION'] == 'Glomerular filtration rate [Volume Rate/Area] in Serum or Plasma by Creatinine-based formula (MDRD)/1.73 sq M']['CODE'].unique()

<StringArray>
['33914-3']
Length: 1, dtype: str

In [44]:
observations[['ENCOUNTER','CATEGORY','UNITS']].isna().sum()

ENCOUNTER    144081
CATEGORY     144081
UNITS        620186
dtype: int64

In [45]:
print("CODES:", observations['CODE'].nunique())
print("DESCRIPTIONS:", observations['DESCRIPTION'].nunique())
print("Codes-description", observations[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 288
DESCRIPTIONS: 289
Codes-description 289


In [46]:
observations.groupby('CODE')['DESCRIPTION'].nunique().sort_values(ascending=False).head(2)

CODE
6299-2     2
92142-9    1
Name: DESCRIPTION, dtype: int64

In [47]:
observations[observations['CODE'] == '6299-2']['DESCRIPTION'].unique()

<StringArray>
['Urea nitrogen [Mass/volume] in Blood', 'Urea Nitrogen [Mass/volume] in Blood']
Length: 2, dtype: str

In [48]:
observations['CATEGORY'].value_counts()

CATEGORY
laboratory        830825
survey            692807
vital-signs       587054
social-history     61678
exam                8200
imaging             2028
procedure           1686
therapy              279
Name: count, dtype: int64

In [49]:
observations['TYPE'].value_counts()

TYPE
numeric    1505829
text        822809
Name: count, dtype: int64

In [50]:
(observations['ENCOUNTER'].isna() == observations['CATEGORY'].isna()).all()

np.True_

In [51]:
pd.crosstab(observations['TYPE'], observations['UNITS'].isna())

UNITS,False,True
TYPE,,
numeric,1505772,57
text,202680,620129


In [52]:
observations[(observations['TYPE'] == 'numeric') & observations["UNITS"].isna()]['DESCRIPTION'].value_counts()

DESCRIPTION
Position of body and posture (observable entity)    19
Cobb angle (observable entity)                      19
Risser sign (finding)                               19
Name: count, dtype: int64

In [53]:
observations[(observations['TYPE'] == 'text') & observations['UNITS'].notna()]['DESCRIPTION'].value_counts()

DESCRIPTION
What was your best estimate of the total income of all family members from all sources  before taxes  in last year [PhenX]    28756
Ketones [Presence] in Urine by Test strip                                                                                     15489
Nitrite [Presence] in Urine by Test strip                                                                                     15489
Hemoglobin [Presence] in Urine by Test strip                                                                                  15489
Leukocyte esterase [Presence] in Urine by Test strip                                                                          15489
Appearance of Urine                                                                                                           14892
Clarity of Urine                                                                                                              14792
Color of Urine                                                  

**One row:** one measurement or answer for one patient. The key `PATIENT, ENCOUNTER, CODE, DATE` is not unique: 4,756 rows repeat a key
(518 of them are full copies). Rows sharing a key differ in VALUE and sometimes
in UNITS; most of them (6,921 rows) are one lab test (CODE `33914-3`,
glomerular filtration rate).

**Missing values:**
- UNITS is missing in 620,186 rows: 620,129 with TYPE `text` and 57 with TYPE `numeric`.
  The 57 numeric rows are three measurements, 19 rows each: "Position of body and
  posture", "Cobb angle" and "Risser sign".
- 202,680 `text` rows do have a unit. The largest groups are urine tests
  (e.g. "Color of Urine", "Glucose [Presence] in Urine by Test strip", about 15,000
  rows each) and one income question (28,756 rows).
- ENCOUNTER and CATEGORY are both missing in the same 144,081 rows (\~6%).

**Codes:** CODE `6299-2` has two descriptions that differ only in letter case
("Urea nitrogen" / "Urea Nitrogen").

**Note:** VALUE holds numbers (TYPE `numeric`, 1.5M rows) and text
(TYPE `text`, 0.8M rows). \~30% of rows have CATEGORY `survey`.

## Procedures

In [54]:
procedures = pd.read_csv('../data/raw/csv/procedures.csv')

print(procedures.shape)
procedures.head(1)

(454904, 10)


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION,BASE_COST,REASONCODE,REASONDESCRIPTION
0,1925-09-30T16:40:18Z,1925-09-30T16:55:18Z,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-f9e0-52523777fdca,SNOMED-CT,430193006,Medication reconciliation (procedure),215.7,NaN,NaN


In [55]:
procedures.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 454904 entries, 0 to 454903
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   START              454904 non-null  str    
 1   STOP               454904 non-null  str    
 2   PATIENT            454904 non-null  str    
 3   ENCOUNTER          454904 non-null  str    
 4   SYSTEM             454904 non-null  str    
 5   CODE               454904 non-null  str    
 6   DESCRIPTION        454904 non-null  str    
 7   BASE_COST          454904 non-null  float64
 8   REASONCODE         139200 non-null  float64
 9   REASONDESCRIPTION  139200 non-null  str    
dtypes: float64(2), str(8)
memory usage: 34.7 MB


In [56]:
procedures[['START','STOP','PATIENT','CODE','ENCOUNTER']].duplicated().sum()

np.int64(0)

In [57]:
procedures['SYSTEM'].value_counts()

SYSTEM
SNOMED-CT    418784
CDT           36120
Name: count, dtype: int64

In [58]:
print("CODES:", procedures['CODE'].nunique())
print("DESCRIPTIONS:", procedures['DESCRIPTION'].nunique())
print("Code-description", procedures[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 744
DESCRIPTIONS: 891
Code-description 891


In [59]:
n_desc = procedures.groupby('CODE')['DESCRIPTION'].nunique()
n_desc.value_counts()

DESCRIPTION
1    597
2    147
Name: count, dtype: int64

In [60]:
multi_codes = n_desc[n_desc > 1].index
procedures[procedures['CODE'].isin(multi_codes)][['CODE','DESCRIPTION','SYSTEM']].drop_duplicates().sort_values(by=['CODE'],ascending=False).head(20)

,CODE,DESCRIPTION,SYSTEM
14751,D9999,unspecified adjunctive procedure by report,CDT
351,D9999,unspecified adjunctive procedure by report,CDT
10079,D9952,occlusal adjustment complete,CDT
235892,D9952,occlusal adjustment - complete,CDT
259443,D9930,treatment of complications (post-surgical) - unusual circumstances by report,CDT
403900,D9930,treatment of complications postsurgical unusual circumstances by report,CDT
209071,D9920,behavior management by report,CDT
100147,D9920,behavior management by report,CDT
122813,D9612,therapeutic parenteral drugs two or more administrations different medications,CDT
35314,D9612,therapeutic parenteral drugs two or more administrations different medications,CDT


In [61]:
multi = procedures[procedures['CODE'].isin(multi_codes)][['CODE','DESCRIPTION']].drop_duplicates()
multi['NORM'] = multi['DESCRIPTION'].str.lower().str.replace(r'[^a-z0-9]', '', regex=True)
multi.groupby('CODE')['NORM'].nunique().max()

np.int64(1)

In [62]:
procedures[procedures['CODE'].isin(multi_codes)]['SYSTEM'].value_counts()

SYSTEM
CDT    26644
Name: count, dtype: int64

In [63]:
procedures['DESCRIPTION'].value_counts()

DESCRIPTION
Depression screening (procedure)                                              52292
Medication reconciliation (procedure)                                         29233
Assessment of health and social care needs (procedure)                        28782
Assessment of substance use (procedure)                                       22506
Patient referral for dental care (procedure)                                  22055
                                                                              ...  
reline maxillary partial denture indirect                                         1
crown  full cast noble metal                                                      1
removal of foreign body from mucosa  skin  or subcutaneous alveolar tissue        1
precision attachment  by report                                                   1
osteotomy - mandibular rami with bone graft; includes obtaining the graft         1
Name: count, Length: 891, dtype: int64

In [64]:
procedures[procedures['SYSTEM'] != 'SNOMED-CT'][['SYSTEM', 'CODE', 'DESCRIPTION']].drop_duplicates()

,SYSTEM,CODE,DESCRIPTION
6,CDT,D2391,resinbased composite one surface posterior
15,CDT,D0120,periodic oral evaluation established patient
16,CDT,D1310,nutritional counseling for control of dental disease
17,CDT,D0603,caries risk assessment and documentation with a finding of high risk
19,CDT,D1208,topical application of fluoride excluding varnish
...,...,...,...
441294,CDT,D2792,crown full cast noble metal
441382,CDT,D7550,partial ostectomy/sequestrectomy for removal of nonvital bone
442995,CDT,D7530,removal of foreign body from mucosa skin or subcutaneous alveolar tissue
453234,CDT,D5862,precision attachment by report


**One row:** one procedure for one patient at one encounter
(`PATIENT, ENCOUNTER, CODE, START, STOP` is unique).

**Missing values:** REASONCODE and REASONDESCRIPTION are each missing in \~69% of rows.

**Codes:**
- 36,120 rows (\~8%) have SYSTEM `CDT` (dental procedures); the rest are `SNOMED-CT`.
- 744 codes, 891 descriptions. 147 codes have two descriptions; all are `CDT`
  and differ only in punctuation and spacing (e.g. "occlusal adjustment - complete"
  / "occlusal adjustment  complete"). These codes cover 26,644 rows, \~74% of
  `CDT` rows.

**Note:** the most frequent descriptions are screenings and assessments
(e.g. "Depression screening (procedure)").

## Imaging studies

In [65]:
imaging_studies = pd.read_csv("../data/raw/csv/imaging_studies.csv")

print(imaging_studies.shape)
imaging_studies.head(1)

(117837, 13)


,Id,DATE,PATIENT,ENCOUNTER,SERIES_UID,BODYSITE_CODE,BODYSITE_DESCRIPTION,MODALITY_CODE,MODALITY_DESCRIPTION,INSTANCE_UID,SOP_CODE,SOP_DESCRIPTION,PROCEDURE_CODE
0,ba419d35-0dfe-8af7-dc0f-1114f3442f00,1932-05-26T16:40:18Z,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-2350-f222d85579ee,1.2.840.99999999.1.42697993.1186557581166,344001,Ankle region structure (body structure),DX,Digital Radiography,1.2.840.99999999.1.1.91405530.1186557581166,1.2.840.10008.5.1.4.1.1.1.1,Digital X-Ray Image Storage,1290459008


In [66]:
imaging_studies.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 117837 entries, 0 to 117836
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   Id                    117837 non-null  str  
 1   DATE                  117837 non-null  str  
 2   PATIENT               117837 non-null  str  
 3   ENCOUNTER             117837 non-null  str  
 4   SERIES_UID            117837 non-null  str  
 5   BODYSITE_CODE         117837 non-null  int64
 6   BODYSITE_DESCRIPTION  117837 non-null  str  
 7   MODALITY_CODE         117837 non-null  str  
 8   MODALITY_DESCRIPTION  117837 non-null  str  
 9   INSTANCE_UID          117837 non-null  str  
 10  SOP_CODE              117837 non-null  str  
 11  SOP_DESCRIPTION       117837 non-null  str  
 12  PROCEDURE_CODE        117837 non-null  int64
dtypes: int64(2), str(11)
memory usage: 11.7 MB


In [67]:
print('Unique Id:', imaging_studies['Id'].nunique())
print("Unique series:", imaging_studies['SERIES_UID'].nunique())
print("Unique instances:", imaging_studies['INSTANCE_UID'].nunique())

Unique Id: 4648
Unique series: 4648
Unique instances: 117837


In [68]:
imaging_studies['MODALITY_DESCRIPTION'].value_counts()

MODALITY_DESCRIPTION
Computed Tomography       110719
Ophthalmic Photography      3608
Ophthalmic Tomography       1884
Digital Radiography          904
Computed Radiography         416
Ultrasound                   306
Name: count, dtype: int64

In [69]:
print("Codes:", imaging_studies['MODALITY_CODE'].nunique())
print("Descriptions:", imaging_studies['MODALITY_DESCRIPTION'].nunique())
print("Code-description pairs:", imaging_studies[['MODALITY_CODE', 'MODALITY_DESCRIPTION']].drop_duplicates().shape[0])

Codes: 6
Descriptions: 6
Code-description pairs: 6


In [70]:
imaging_studies.groupby('MODALITY_DESCRIPTION')['Id'].nunique()

MODALITY_DESCRIPTION
Computed Radiography       416
Computed Tomography        276
Digital Radiography        904
Ophthalmic Photography    1804
Ophthalmic Tomography      942
Ultrasound                 306
Name: Id, dtype: int64

In [71]:
imaging_studies.groupby('Id')['SERIES_UID'].nunique().max()

np.int64(1)

**One row:** one image (`INSTANCE_UID` is unique): 117,837 images from
4,648 studies (unique `Id`), one series per study.

**Missing values:** none.

**Codes:** 6 modality codes, each with exactly one description.

**Note:** "Computed Tomography" is \~94% of rows but only \~6% of studies,
with about 400 images per study. The most common study type is
"Ophthalmic Photography" (\~39% of studies, two images per study on average).

## Immunizations

In [72]:
immunizations = pd.read_csv("../data/raw/csv/immunizations.csv")
print(immunizations.shape)
immunizations.head(1)

(61740, 6)


,DATE,PATIENT,ENCOUNTER,CODE,DESCRIPTION,BASE_COST
0,1945-09-12T16:40:18Z,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-7164-ded4779a9637,140,Influenza split virus trivalent PF,136.0


In [73]:
immunizations.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 61740 entries, 0 to 61739
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATE         61740 non-null  str    
 1   PATIENT      61740 non-null  str    
 2   ENCOUNTER    61740 non-null  str    
 3   CODE         61740 non-null  int64  
 4   DESCRIPTION  61740 non-null  str    
 5   BASE_COST    61740 non-null  float64
dtypes: float64(1), int64(1), str(4)
memory usage: 2.8 MB


In [74]:
immunizations[['PATIENT','ENCOUNTER','CODE','DATE']].duplicated().sum()

np.int64(0)

In [75]:
print("CODES:", immunizations['CODE'].nunique())
print("Descriptions:", immunizations['DESCRIPTION'].nunique())
print("Code-description pairs:", immunizations[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 22
Descriptions: 25
Code-description pairs: 25


In [76]:
n_desc = immunizations.groupby('CODE')['DESCRIPTION'].nunique()
n_desc.value_counts()

DESCRIPTION
1    19
2     3
Name: count, dtype: int64

In [77]:
multi_codes = n_desc[n_desc > 1].index
immunizations[immunizations['CODE'].isin(multi_codes)][['CODE','DESCRIPTION']].drop_duplicates().sort_values(['CODE'],ascending=False)

,CODE,DESCRIPTION
276,133,Pneumococcal conjugate PCV 13
15057,133,pneumococcal conjugate vaccine 13 valent
307,115,Tdap
15055,115,tetanus toxoid reduced diphtheria toxoid and acellular pertussis vaccine adsorbed
310,114,meningococcal MCV4P
15056,114,meningococcal polysaccharide (groups A C Y and W-135) diphtheria toxoid conjugate vaccine (MCV4P)


**One row:** one vaccination for one patient at one encounter
(`PATIENT, ENCOUNTER, CODE, DATE` is unique).

**Missing values:** none.

**Codes:** 22 codes, 25 descriptions. 3 codes (114, 115, 133) have a short and
a long description of the same vaccine (e.g. "Tdap" / "tetanus toxoid  reduced
diphtheria toxoid  and acellular pertussis vaccine  adsorbed"). The long
descriptions contain double spaces.

## Payer transitions

In [78]:
payer_transitions = pd.read_csv("../data/raw/csv/payer_transitions.csv")
print(payer_transitions.shape)
payer_transitions.head(1)

(41902, 8)


,PATIENT,MEMBERID,START_DATE,END_DATE,PAYER,SECONDARY_PAYER,PLAN_OWNERSHIP,OWNER_NAME
0,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-8b26-cf0adb43a362,1969-10-01T16:40:18Z,1970-09-30T16:40:18Z,df166300-5a78-3502-a46a-832842197811,NaN,Self,Corie618 Sofia418 Mante251


In [79]:
payer_transitions.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 41902 entries, 0 to 41901
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   PATIENT          41902 non-null  str  
 1   MEMBERID         31849 non-null  str  
 2   START_DATE       41902 non-null  str  
 3   END_DATE         41902 non-null  str  
 4   PAYER            41902 non-null  str  
 5   SECONDARY_PAYER  3940 non-null   str  
 6   PLAN_OWNERSHIP   31849 non-null  str  
 7   OWNER_NAME       31849 non-null  str  
dtypes: str(8)
memory usage: 2.6 MB


In [80]:
payer_transitions[['PATIENT','START_DATE','END_DATE']].duplicated().sum()

np.int64(0)

In [81]:
payer_transitions['MEMBERID'].isna().sum()

np.int64(10053)

In [82]:
payer_transitions[payer_transitions['MEMBERID'].isna()]['PAYER'].value_counts()

PAYER
e03e23c9-4df1-3eb6-a62d-f70f02301496    10053
Name: count, dtype: int64

In [83]:
((payer_transitions['MEMBERID'].isna() == payer_transitions['PLAN_OWNERSHIP'].isna())
& (payer_transitions['MEMBERID'].isna() == payer_transitions['OWNER_NAME'].isna())).all()

np.True_

**One row:** one insurance period for one patient
(`PATIENT, START_DATE, END_DATE` is unique).

**Missing values:**
- MEMBERID, PLAN_OWNERSHIP and OWNER_NAME are empty in the same 10,053 rows,
  all with the same PAYER (`e03e23c9-4df1-3eb6-a62d-f70f02301496`, see Payers).
- SECONDARY_PAYER is filled in only 3,940 rows.

## Supplies

In [84]:
supplies = pd.read_csv('../data/raw/csv/supplies.csv')
print(supplies.shape)
supplies.head(1)

(21235, 6)


,DATE,PATIENT,ENCOUNTER,CODE,DESCRIPTION,QUANTITY
0,1962-12-19,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-90da-4142cf7c5857,337388004,Blood glucose testing strips (physical object),50


In [85]:
supplies.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 21235 entries, 0 to 21234
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   DATE         21235 non-null  str  
 1   PATIENT      21235 non-null  str  
 2   ENCOUNTER    21235 non-null  str  
 3   CODE         21235 non-null  int64
 4   DESCRIPTION  21235 non-null  str  
 5   QUANTITY     21235 non-null  int64
dtypes: int64(2), str(4)
memory usage: 995.5 KB


In [86]:
supplies[['DATE','PATIENT','ENCOUNTER','CODE']].duplicated().sum()

np.int64(253)

In [87]:
dup = supplies[supplies.duplicated(subset=['CODE','PATIENT','ENCOUNTER','DATE'], keep=False)]
dup.groupby(['CODE','PATIENT','ENCOUNTER','DATE']).nunique().max()

DESCRIPTION    1
QUANTITY       2
dtype: int64

In [88]:
print("CODES:", supplies['CODE'].nunique())
print("Descriptions:", supplies['DESCRIPTION'].nunique())
print("Codes-descriptions pairs: ", supplies[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 31
Descriptions: 31
Codes-descriptions pairs:  31


**One row:** one supply item used for one patient at one encounter.
The key `PATIENT, ENCOUNTER, CODE, DATE` is not unique: 253 rows repeat a key.
187 of them are full copies; in the remaining 66 only QUANTITY differs.

**Missing values:** none.

**Codes:** 31 codes, each with exactly one DESCRIPTION.

## Careplans

In [89]:
careplans = pd.read_csv('../data/raw/csv/careplans.csv')
print(careplans.shape)
careplans.head(1)

(10098, 9)


,Id,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,REASONCODE,REASONDESCRIPTION
0,ba419d35-0dfe-8af7-4cb7-df3664b828e3,1929-10-23,1930-09-10,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-a7b9-4f099a8e2782,53950000,Respiratory therapy (procedure),NaN,NaN


In [90]:
careplans.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 10098 entries, 0 to 10097
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Id                 10098 non-null  str    
 1   START              10098 non-null  str    
 2   STOP               8245 non-null   str    
 3   PATIENT            10098 non-null  str    
 4   ENCOUNTER          10098 non-null  str    
 5   CODE               10098 non-null  int64  
 6   DESCRIPTION        10098 non-null  str    
 7   REASONCODE         4270 non-null   float64
 8   REASONDESCRIPTION  4270 non-null   str    
dtypes: float64(1), int64(1), str(7)
memory usage: 710.1 KB


In [91]:
careplans['Id'].is_unique

True

In [92]:
print("CODES:", careplans['CODE'].nunique())
print("Descriptions:", careplans['DESCRIPTION'].nunique())
print("Codes-descriptions:", careplans[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 37
Descriptions: 37
Codes-descriptions: 37


**One row:** one care plan for one patient.

**Missing values:** STOP is missing in 1,853 rows (plan still
active); REASONCODE and REASONDESCRIPTION are filled in only 4,270 rows.

**Codes:** 37 codes, each with exactly one DESCRIPTION.

## Devices

In [93]:
devices = pd.read_csv("../data/raw/csv/devices.csv")
print(devices.shape)
devices.head(1)

(3266, 7)


,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,UDI
0,1932-05-26T16:40:18Z,1990-02-06T17:26:04Z,ba419d35-0dfe-8af7-347c-eebf02485a56,ba419d35-0dfe-8af7-2350-f222d85579ee,228869008,Manual wheelchair (physical object),(01)63905317816316(11)320505(17)570520(10)5527281(21)31284


In [94]:
devices.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 3266 entries, 0 to 3265
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   START        3266 non-null   str  
 1   STOP         2297 non-null   str  
 2   PATIENT      3266 non-null   str  
 3   ENCOUNTER    3266 non-null   str  
 4   CODE         3266 non-null   int64
 5   DESCRIPTION  3266 non-null   str  
 6   UDI          3266 non-null   str  
dtypes: int64(1), str(6)
memory usage: 178.7 KB


In [95]:
devices[['START','STOP','PATIENT','CODE']].duplicated().sum()

np.int64(0)

In [96]:
print('CODES:', devices['CODE'].nunique())
print("Descriptions:", devices['DESCRIPTION'].nunique())
print('Codes-description pairs:', devices[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 26
Descriptions: 26
Codes-description pairs: 26


In [97]:
devices.groupby('PATIENT').size().describe()

count    780.000000
mean       4.187179
std       16.174461
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max      393.000000
dtype: float64

In [98]:
devices[devices['PATIENT'] == 'cc416652-c144-cbaf-9bbe-1c9cc039b0b1'][['DESCRIPTION','STOP']]

,DESCRIPTION,STOP
1019,Manual wheelchair (physical object),1917-03-26T18:50:42Z
1020,Manual wheelchair (physical object),1927-12-12T02:51:42Z
1021,Blood glucose meter (physical object),NaN
1022,Respiratory apnea monitoring system (physical object),1971-04-28T00:13:32Z
1023,Polysomnography analyzer (physical object),1971-04-28T00:13:32Z
...,...,...
1407,Respiratory humidifier (physical object),2020-11-17T19:16:59Z
1408,Home continuous positive airway pressure unit (physical object),2025-12-16T19:20:11Z
1409,Respiratory humidifier (physical object),2025-12-16T19:20:11Z
1410,Home continuous positive airway pressure unit (physical object),NaN


**One row:** one device used by one patient
(`PATIENT, CODE, START, STOP` is unique).

**Missing values:** STOP has 2,297 non-null values out of 3,266
(device still in use).

**Codes:** 26 codes, each with exactly one DESCRIPTION.

**Note:** 780 patients have at least one device; the median is 2 devices per
patient, but one patient has 393 rows: the same devices (e.g. "Home continuous
positive airway pressure unit", "Respiratory humidifier") are recorded many
times with different STOP dates.

## Allergies

In [99]:
allergies = pd.read_csv('../data/raw/csv/allergies.csv')
print(allergies.shape)
allergies.head(1)

(1283, 15)


,START,STOP,PATIENT,ENCOUNTER,CODE,SYSTEM,DESCRIPTION,TYPE,CATEGORY,REACTION1,DESCRIPTION1,SEVERITY1,REACTION2,DESCRIPTION2,SEVERITY2
0,1926-08-12,NaN,fac73443-4601-476e-bfb3-95890a9bc2d9,fac73443-4601-476e-6bd8-e0fefc2ca72f,609328004,Unknown,Allergic disposition (finding),allergy,environment,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
allergies.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1283 entries, 0 to 1282
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   START         1283 non-null   str    
 1   STOP          0 non-null      float64
 2   PATIENT       1283 non-null   str    
 3   ENCOUNTER     1283 non-null   str    
 4   CODE          1283 non-null   int64  
 5   SYSTEM        1283 non-null   str    
 6   DESCRIPTION   1283 non-null   str    
 7   TYPE          1283 non-null   str    
 8   CATEGORY      1283 non-null   str    
 9   REACTION1     572 non-null    float64
 10  DESCRIPTION1  572 non-null    str    
 11  SEVERITY1     572 non-null    str    
 12  REACTION2     342 non-null    float64
 13  DESCRIPTION2  342 non-null    str    
 14  SEVERITY2     342 non-null    str    
dtypes: float64(3), int64(1), str(11)
memory usage: 150.5 KB


In [101]:
allergies[['START','PATIENT','CODE']].duplicated().sum() #STOP got 0 values

np.int64(0)

In [102]:
allergies['SYSTEM'].value_counts()

SYSTEM
Unknown    1283
Name: count, dtype: int64

In [103]:
print("CODES:", allergies['CODE'].nunique())
print("Descriptions:", allergies['DESCRIPTION'].nunique())
print("Code-description pair:", allergies[['CODE','DESCRIPTION']].drop_duplicates().shape[0])

CODES: 21
Descriptions: 21
Code-description pair: 21


**One row:** one allergy or intolerance for one patient
(`PATIENT, CODE, START` is unique).

**Missing values:**
- STOP is empty in all 1,283 rows.
- Reaction columns are optional: REACTION1/DESCRIPTION1/SEVERITY1 are filled in
  572 rows, REACTION2/DESCRIPTION2/SEVERITY2 in 342 rows.

**Codes:** SYSTEM is "Unknown" in all rows. 21 codes, each with exactly one description.

## Providers

In [104]:
providers = pd.read_csv("../data/raw/csv/providers.csv")
print(providers.shape)
providers.head(1)

(858, 13)


,Id,ORGANIZATION,NAME,GENDER,SPECIALITY,ADDRESS,CITY,STATE,ZIP,LAT,LON,ENCOUNTERS,PROCEDURES
0,8d32509a-2844-3440-aac1-59efe106054b,74ab949d-17ac-3309-83a0-13b4405c66aa,Gabriel934 Reilly981,F,GENERAL PRACTICE,881 Main Street,Fitchburg,MA,1420,42.586487,-71.80521,2076,0


In [105]:
providers.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            858 non-null    str    
 1   ORGANIZATION  858 non-null    str    
 2   NAME          858 non-null    str    
 3   GENDER        858 non-null    str    
 4   SPECIALITY    858 non-null    str    
 5   ADDRESS       858 non-null    str    
 6   CITY          858 non-null    str    
 7   STATE         858 non-null    str    
 8   ZIP           858 non-null    int64  
 9   LAT           858 non-null    float64
 10  LON           858 non-null    float64
 11  ENCOUNTERS    858 non-null    int64  
 12  PROCEDURES    858 non-null    int64  
dtypes: float64(2), int64(3), str(8)
memory usage: 87.3 KB


In [106]:
providers['Id'].is_unique

True

In [107]:
providers['ZIP'].astype(str).str.len().value_counts()

ZIP
8    811
4     47
Name: count, dtype: int64

In [108]:
providers['ORGANIZATION'].is_unique

True

**One row:** one provider.

**Note:** ZIP is stored as a number with 8 digits (811 rows) or 4 digits
(47 rows), e.g. `1420`, `21302331`.


## Organizations

In [109]:
organizations = pd.read_csv("../data/raw/csv/organizations.csv")
print(organizations.shape)
organizations.head(1)

(858, 11)


,Id,NAME,ADDRESS,CITY,STATE,ZIP,LAT,LON,PHONE,REVENUE,UTILIZATION
0,74ab949d-17ac-3309-83a0-13b4405c66aa,Fitchburg Outpatient Clinic,881 Main Street,Fitchburg,MA,1420,42.586487,-71.80521,978-342-9781 Or 978-342-9781,0.0,2076


In [110]:
organizations.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Id           858 non-null    str    
 1   NAME         858 non-null    str    
 2   ADDRESS      858 non-null    str    
 3   CITY         858 non-null    str    
 4   STATE        858 non-null    str    
 5   ZIP          858 non-null    int64  
 6   LAT          858 non-null    float64
 7   LON          858 non-null    float64
 8   PHONE        858 non-null    str    
 9   REVENUE      858 non-null    float64
 10  UTILIZATION  858 non-null    int64  
dtypes: float64(3), int64(2), str(6)
memory usage: 73.9 KB


In [111]:
organizations['Id'].is_unique

True

In [112]:
organizations['ZIP'].astype(str).str.len().value_counts()

ZIP
8    811
4     47
Name: count, dtype: int64

**One row:** one organization.

**Note:** same ZIP format as in providers (8 digits in 811 rows, 4 digits in 47).

## Payers

In [113]:
payers = pd.read_csv('../data/raw/csv/payers.csv')
print(payers.shape)
payers.head(1)

(10, 22)


,Id,NAME,OWNERSHIP,ADDRESS,CITY,STATE_HEADQUARTERED,ZIP,PHONE,AMOUNT_COVERED,AMOUNT_UNCOVERED,...,UNCOVERED_ENCOUNTERS,COVERED_MEDICATIONS,UNCOVERED_MEDICATIONS,COVERED_PROCEDURES,UNCOVERED_PROCEDURES,COVERED_IMMUNIZATIONS,UNCOVERED_IMMUNIZATIONS,UNIQUE_CUSTOMERS,QOLS_AVG,MEMBER_MONTHS
0,a735bf55-83e9-331a-899d-a82a60b9f60c,Medicare,GOVERNMENT,NaN,NaN,NaN,NaN,NaN,79008092.36,4747440.1,...,0,40399,0,63285,0,6489,0,347,0.732904,66480


In [114]:
payers.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Id                       10 non-null     str    
 1   NAME                     10 non-null     str    
 2   OWNERSHIP                10 non-null     str    
 3   ADDRESS                  0 non-null      float64
 4   CITY                     0 non-null      float64
 5   STATE_HEADQUARTERED      0 non-null      float64
 6   ZIP                      0 non-null      float64
 7   PHONE                    0 non-null      float64
 8   AMOUNT_COVERED           10 non-null     float64
 9   AMOUNT_UNCOVERED         10 non-null     float64
 10  REVENUE                  10 non-null     float64
 11  COVERED_ENCOUNTERS       10 non-null     int64  
 12  UNCOVERED_ENCOUNTERS     10 non-null     int64  
 13  COVERED_MEDICATIONS      10 non-null     int64  
 14  UNCOVERED_MEDICATIONS    10 non-null    

In [115]:
payers['Id'].is_unique

True

In [116]:
payers[payers['Id'] == 'e03e23c9-4df1-3eb6-a62d-f70f02301496'][['Id','NAME']]

,Id,NAME
9,e03e23c9-4df1-3eb6-a62d-f70f02301496,NO_INSURANCE


**One row:** one payer (10 payers overall in the table).

**Missing values:**
- ADDRESS, CITY, STATE_HEADQUARTERED, ZIP and PHONE are empty in all 10 rows.

**Note:** `e03e23c9-4df1-3eb6-a62d-f70f02301496` is NO_INSURANCE.
This is the PAYER of all 10,053 rows in payer_transitions where MEMBERID,
PLAN_OWNERSHIP and OWNER_NAME are empty.